# **Population Structure Analysis – PopPUNK**

## **Tool Information**

- **Tool:** PopPUNK v2.6.3
- **Input:** Assembled genome contigs (FASTA)
- **Organism:** *Acinetobacter baumannii*
- **Analysis type:** Population structure analysis and lineage clustering

PopPUNK (Population Partitioning Using Nucleotide K-mers) is a scalable tool designed to analyse bacterial population structure using whole genome assemblies. It estimates pairwise core and accessory genome distances using k-mer comparisons and clusters genomes into genetically related lineages.

This notebook documents the construction of a PopPUNK database and clustering analysis of *Acinetobacter baumannii* genome assemblies.

## **Installation**

### 1) Create a dedicated environment
We create a dedicated conda environment to isolate PopPUNK and its dependencies from other tools. This ensures a stable and reproducible setup for bacterial population analysis.

In [ ]:
%%bash

conda create -n poppunk_aba

### 2) Install PopPUNK using mamba
We install PopPUNK using mamba to efficiently resolve dependencies and speed up installation. The tool is installed from Bioconda along with required packages from conda-forge.

In [ ]:
%%bash

source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate poppunk_aba

conda install -n base -c conda-forge mamba -y
mamba install -c bioconda -c conda-forge poppunk -y

### 3) Verify Installation
We verify that PopPUNK is installed correctly by checking its version. This confirms that the tool is ready for population clustering and genomic analysis.

In [ ]:
%%bash

source /home/anaconda/miniconda3/etc/profile.d/conda.sh
conda activate poppunk_aba

poppunk --version

poppunk 2.6.3


## **Input List**

PopPUNK requires a two-column tab-separated file containing:

1. **Sample name**
2. **Full path to genome assembly file**

In this step, we generate `fasta_list.txt`, which maps each assembly file to its corresponding sample identifier.

In [ ]:
%%bash 

# Define the output list file
INPUT_LIST="/data/internship_data/nidhi/aba/new_output/poppunk_output/fasta_list.txt"

# Loop through the files to create the name-path mapping
for file in /data/internship_data/nidhi/aba/new_output/nextflow_output/assemblies/*.fasta; do
    # Extract the filename without the path and extension as the sample name
    sample_name=$(basename "$file" .fasta)
    # Append to the list: SampleName [TAB] Path
    printf "%s\t%s\n" "$sample_name" "$file" >> $INPUT_LIST
done

echo "Input list created at $INPUT_LIST"


Input list created at /data/internship_data/nidhi/aba/new_output/poppunk_output/fasta_list.txt


## **Database Installation and Initialisation**

Database construction is performed during runtime using the `--create-db` option. This step calculates pairwise core and accessory genome distances and stores them in a PopPUNK reference database directory.

⚠️ Run the database creation step only once per dataset. Re-running it will overwrite existing distance matrices unless a new output directory is specified.

## **Execution Strategy**

PopPUNK was executed using a parallelised workflow to efficiently process multiple genome assemblies. All genomes were included in a single distance calculation step to ensure accurate estimation of core and accessory genomic similarity across the entire dataset.

A multi-threaded strategy was used to accelerate k-mer comparison and model fitting steps. Sixteen CPU threads were assigned to balance computational efficiency and server resource availability.

### i) Model Fitting Using DBSCAN (`--fit-model dbscan`)

After database construction, PopPUNK calculates pairwise core and accessory genome distances for all isolates. These distances are plotted in a two-dimensional space (core vs accessory divergence).

The `--fit-model dbscan` step applies the DBSCAN clustering algorithm to:

- Identify dense groups of closely related genomes  
- Separate within-lineage from between-lineage distances  
- Assign preliminary cluster (lineage) labels  
- Detect potential outliers  

DBSCAN is advantageous because it does not require specifying the number of clusters and can identify noise or highly divergent isolates. This step produces the initial lineage structure of the dataset.

### ii) Cluster Refinement (`--fit-model refine`)

The refinement step improves clustering accuracy by optimising the boundaries defined during DBSCAN fitting.

Why it's needed:

- Re-evaluates cluster connectivity  
- Adjusts decision thresholds  
- Resolves borderline genomes  
- Minimises over- or under-clustering  

This step produces the final, high-confidence cluster assignments used for downstream population structure analysis and interpretation.

In [ ]:
%%bash

# Conda Activation
source /home/anaconda/miniconda3/etc/profile.d/conda.sh

# Activate PopPUNK environment
conda activate poppunk_aba

# Paths
FASTA_LIST="/data/internship_data/nidhi/aba/new_output/poppunk_output/fasta_list.txt"
OUTPUT_DIR="/data/internship_data/nidhi/aba/new_output/poppunk_output"
DB_NAME="query_assignment"
THREADS=16

# PopPUNK database creation

poppunk --create-db \
    --r-files ${FASTA_LIST} \
    --output ${OUTPUT_DIR}/${DB_NAME} \
    --model dbscan \
    --threads ${THREADS}

poppunk \
--fit-model dbscan \
--ref-db  /data/internship_data/nidhi/aba/new_output/poppunk_output/query_assignment \
--threads 16

poppunk \
    --fit-model refine \
    --ref-db /data/internship_data/nidhi/aba/new_output/poppunk_output/query_assignment \
    --threads 16


## **Visualisation and Network Generation**

After final cluster refinement, visualisation files were generated using `poppunk_visualise` to enable graphical exploration of population structure and lineage relationships.

This step creates multiple output formats compatible with commonly used phylogenetic and network visualisation platforms.

### Key Parameters Used:

- `--tree both`  
  Generates both neighbour-joining and minimum spanning tree (MST) representations.

- `--mst-distances core`  
  Constructs the minimum spanning tree using core genome distances.

- `--network-file`  
  Uses the refined PopPUNK graph file (`.gt`) generated during clustering.

- `--microreact`  
  Produces files compatible with Microreact for interactive online visualisation.

- `--cytoscape`  
  Generates network files for Cytoscape-based network analysis.

- `--grapetree`  
  Outputs files compatible with GrapeTree for MST visualisation.

- `--phandango`  
  Produces files for Phandango visualisation of tree and metadata integration.

- `--ref-db`  
  Specifies the reference PopPUNK database.

- `--output`  
  Defines the directory where visualisation outputs are stored.

In [ ]:
%%bash

# Conda Activation
source /home/anaconda/miniconda3/etc/profile.d/conda.sh

# Activate PopPUNK environment
conda activate poppunk_aba

# Visualistion
poppunk_visualise \
    --tree both \
    --mst-distances core \
    --network-file /data/internship_data/nidhi/aba/new_output/poppunk_output/query_assignment/query_assignment_graph.gt \
    --microreact \
    --cytoscape \
    --grapetree \
    --phandango \
    --ref-db /data/internship_data/nidhi/aba/new_output/poppunk_output/query_assignment \
    --output /data/internship_data/nidhi/aba/new_output/poppunk_output/visualisation



## **Output Summary**

### DBSCAN Clustering and Refinement

After running `--fit-model dbscan` and `--fit-model refine`, PopPUNK generates:

- `clusters.csv` → Final cluster assignments per isolate  
- `lineages.csv` → Lineage definitions  
- `query_assignment_graph.gt` → Network graph file  
- Core and accessory distance matrices  

These files define the final genomic clusters used for population structure analysis.

### Visualisation Output

The `poppunk_visualise` step generates:

- Neighbour-joining and minimum spanning tree files  
- Network graph files  
- Microreact, Cytoscape, GrapeTree, and Phandango-compatible outputs  

These files enable graphical exploration of lineage relationships and cluster connectivity within the dataset.

## **Citation**

Lees JA, Harris SR, Tonkin-Hill G, Gladstone RA,
Lo SW, Weiser JN, Corander J, Bentley SD,
Croucher NJ.

Fast and flexible bacterial genomic epidemiology with PopPUNK.

Genome Research. 2019;29(2):304–316.

https://doi.org/10.1101/gr.241455.118